In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas torch

import json
import torch
import pandas as pd

from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit,train_test_split
from sklearn.metrics import accuracy_score
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU available: True
GPU: Tesla T4


In [3]:
DATASET_PATH = "/content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/data/raw/ethan_tickets_10000.csv"

df = pd.read_csv(DATASET_PATH)

print("Total tickets:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Total tickets: 10000
Columns: ['ticket_id', 'text', 'resolver_group', 'priority', 'parent_incident_id', 'client_id', 'caller_name']


,ticket_id,text,resolver_group,priority,parent_incident_id,client_id,caller_name
0,TKT006253,The server reports a hardware failure involvin...,Hardware Support,P3,INC03557,Client_B,User_4770
1,TKT004685,The web server fails to serve pages to externa...,Web Hosting,P3,INC02656,Client_B,User_4558
2,TKT001732,Users receive an installation error when updat...,Software Support,P2,INC00970,Client_B,User_8927
3,TKT004743,The collaboration platform fails to load share...,Email and Collaboration,P3,INC02693,Client_C,User_1042
4,TKT004522,The security system reports an unexpected chan...,IT Security Operations,P3,INC02560,Client_A,User_7891


In [4]:

labels = sorted(df["resolver_group"].unique())

label_mapping = {label: index for index, label in enumerate(labels)}

df["y"] = df["resolver_group"].map(label_mapping)
print(df["y"])

train, test = train_test_split(df, test_size=0.2)


print("Training records:", len(train))
print("Testing records:", len(test))
print("Resolver groups:", len(labels))
print("Labels:", labels)

0       11
1       22
2       18
3       10
4       12
        ..
9995     2
9996     7
9997    15
9998    15
9999     9
Name: y, Length: 10000, dtype: int64
Training records: 8000
Testing records: 2000
Resolver groups: 24
Labels: ['Application Support', 'Backup and Recovery', 'CRM Support', 'Cloud Infrastructure', 'Cybersecurity', 'Data Engineering', 'Database Administration', 'Desktop Support', 'DevOps', 'ERP Support', 'Email and Collaboration', 'Hardware Support', 'IT Security Operations', 'IT Service Desk', 'Identity and Access Management', 'Linux Administration', 'Monitoring and Observability', 'Network Operations', 'Software Support', 'Storage Administration', 'Telecommunications', 'Virtualization Support', 'Web Hosting', 'Windows Administration']


In [5]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Tokenizer loaded successfully


In [6]:
def enc(data):
    dataset = Dataset.from_dict({
        "text": data["text"].tolist(),
        "labels": data["y"].tolist()
    })

    dataset = dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=128
        ),
        batched=True
    )

    return dataset.remove_columns(["text"])


train_dataset = enc(train)
test_dataset = enc(test)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training dataset: 8000
Testing dataset: 2000


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label={i: label for i, label in enumerate(labels)},
    label2id={label: i for i, label in enumerate(labels)}
)

print("Model loaded successfully")
print("Number of resolver groups:", model.config.num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully
Number of resolver groups: 24


In [8]:
from sklearn.metrics import accuracy_score

def compute_metrics(prediction):
    predictions = prediction.predictions.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(
            prediction.label_ids,
            predictions
        )
    }

In [9]:
training_args = TrainingArguments(
    output_dir="/content/out",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

print("Training parameters configured")

Training parameters configured


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer initialized successfully")

Trainer initialized successfully


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.083081,0.030206,1.000000
2,0.022506,0.009826,1.000000
3,0.011686,0.007493,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1500, training_loss=0.3724244318008423, metrics={'train_runtime': 671.7576, 'train_samples_per_second': 35.727, 'train_steps_per_second': 2.233, 'total_flos': 1578978164736000.0, 'train_loss': 0.3724244318008423, 'epoch': 3.0})

In [12]:
results = trainer.evaluate()

print("Accuracy:", results["eval_accuracy"])

Training Loss,Validation Loss,Epoch,Accuracy
0.011686,0.007493,3,1.000000


Accuracy: 1.0


In [14]:
import os
import json

MODEL_OUTPUT = "/content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/models/ethan_resolver_model"

os.makedirs(MODEL_OUTPUT, exist_ok=True)

trainer.save_model(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)

with open(f"{MODEL_OUTPUT}/labels.json", "w") as file:
    json.dump(labels, file)

print("Model saved successfully!")
print("Location:", MODEL_OUTPUT)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Location: /content/drive/MyDrive/EthanGlobal_AI_Triage_Agent/models/ethan_resolver_model
